In [1]:
import findspark

In [2]:
findspark.init("/opt/spark")

In [3]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

In [4]:
spark = SparkSession.builder.appName("SQL Functions").master("local[2]") \
.config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
.config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
.config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
.getOrCreate()

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-core_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-9c35bf1f-84b3-4789-94a7-abee5afe6b79;1.0
	confs: [default]
	found io.delta#delta-core_2.12;2.4.0 in central
	found io.delta#delta-storage;2.4.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-core_2.12/2.4.0/delta-core_2.12-2.4.0.jar ...
	[SUCCESSFUL ] io.delta#delta-core_2.12;2.4.0!delta-core_2.12.jar (2567ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/2.4.0/delta-storage-2.4.0.jar ...
	[SUCCESSFUL ] io.delta#delta-storage;2.4.0!delta-storage.jar (163ms)
downloading https://repo1.maven.org/maven2/org/antlr/antlr4-runtime/4.9.3/antlr4-runtime-4.9.3.jar ...
	[SUCCESSFUL ] org.antlr#antlr4-runtime;4.9.3!antlr4-runtime.jar (338ms)
:: resolution report :: resolve 6122ms :: artifacts dl 3094m

In [6]:
product_groups = spark.createDataFrame([(1, 'Smartphone'),(2, 'Laptop'), (3, 'Tablet')], 
                                       ['group_id', 'group_name'])

product_groups.show()

+--------+----------+
|group_id|group_name|
+--------+----------+
|       1|Smartphone|
|       2|    Laptop|
|       3|    Tablet|
+--------+----------+



In [9]:
products = spark.createDataFrame([(1, 'Microsoft Lumia', 1, 200),
	(2, 'HTC One', 1, 400),
	(3,'Nexus', 1, 500),
	(4, 'iPhone', 1, 900),
	(5, 'HP Elite', 2, 1200),
	(6, 'Lenovo Thinkpad', 2, 700),
	(7, 'Sony VAIO', 2, 700),
	(8, 'Dell Vostro', 2, 800),
	(9, 'iPad', 3, 700),
	(10, 'Kindle Fire', 3, 150),
	(11, 'Samsung Galaxy Tab', 3, 200)], 
['product_id', 'product_name', 'group_id', 'price'])

In [10]:
products.show()

+----------+------------------+--------+-----+
|product_id|      product_name|group_id|price|
+----------+------------------+--------+-----+
|         1|   Microsoft Lumia|       1|  200|
|         2|           HTC One|       1|  400|
|         3|             Nexus|       1|  500|
|         4|            iPhone|       1|  900|
|         5|          HP Elite|       2| 1200|
|         6|   Lenovo Thinkpad|       2|  700|
|         7|         Sony VAIO|       2|  700|
|         8|       Dell Vostro|       2|  800|
|         9|              iPad|       3|  700|
|        10|       Kindle Fire|       3|  150|
|        11|Samsung Galaxy Tab|       3|  200|
+----------+------------------+--------+-----+



In [12]:
products.groupBy('group_id').agg(F.avg('price').alias('avg_price')).show()

+--------+---------+
|group_id|avg_price|
+--------+---------+
|       1|    500.0|
|       2|    850.0|
|       3|    350.0|
+--------+---------+



In [31]:
window_without_order = Window.partitionBy("group_name")

In [32]:
products.join(product_groups, products['group_id'] == product_groups['group_id']) \
.withColumn("group_avg", F.avg("price").over(window_without_order)).show()

+----------+------------------+--------+-----+--------+----------+---------+
|product_id|      product_name|group_id|price|group_id|group_name|group_avg|
+----------+------------------+--------+-----+--------+----------+---------+
|         5|          HP Elite|       2| 1200|       2|    Laptop|    850.0|
|         6|   Lenovo Thinkpad|       2|  700|       2|    Laptop|    850.0|
|         7|         Sony VAIO|       2|  700|       2|    Laptop|    850.0|
|         8|       Dell Vostro|       2|  800|       2|    Laptop|    850.0|
|         1|   Microsoft Lumia|       1|  200|       1|Smartphone|    500.0|
|         2|           HTC One|       1|  400|       1|Smartphone|    500.0|
|         3|             Nexus|       1|  500|       1|Smartphone|    500.0|
|         4|            iPhone|       1|  900|       1|Smartphone|    500.0|
|         9|              iPad|       3|  700|       3|    Tablet|    350.0|
|        10|       Kindle Fire|       3|  150|       3|    Tablet|    350.0|

In [27]:
# The ROW_NUMBER(), RANK(), and DENSE_RANK() functions assign an integer to each row based on its order in its result set.

In [28]:
# The ROW_NUMBER() function assigns a sequential number to each row in each partition.

In [36]:
window_with_order = Window.partitionBy("group_name").orderBy("price")

In [50]:
products.join(product_groups, products['group_id'] == product_groups['group_id']) \
.withColumn("row_number", F.row_number().over(window_with_order)).show()

+----------+------------------+--------+-----+--------+----------+----------+
|product_id|      product_name|group_id|price|group_id|group_name|row_number|
+----------+------------------+--------+-----+--------+----------+----------+
|         6|   Lenovo Thinkpad|       2|  700|       2|    Laptop|         1|
|         7|         Sony VAIO|       2|  700|       2|    Laptop|         2|
|         8|       Dell Vostro|       2|  800|       2|    Laptop|         3|
|         5|          HP Elite|       2| 1200|       2|    Laptop|         4|
|         1|   Microsoft Lumia|       1|  200|       1|Smartphone|         1|
|         2|           HTC One|       1|  400|       1|Smartphone|         2|
|         3|             Nexus|       1|  500|       1|Smartphone|         3|
|         4|            iPhone|       1|  900|       1|Smartphone|         4|
|        10|       Kindle Fire|       3|  150|       3|    Tablet|         1|
|        11|Samsung Galaxy Tab|       3|  200|       3|    Table

In [38]:
# The RANK() function assigns ranking within an ordered partition. If rows have the same values, the  RANK() function assigns the same rank, with the next ranking(s) skipped.

In [49]:
products.join(product_groups, products['group_id'] == product_groups['group_id']) \
.withColumn("rank", F.rank().over(window_with_order)).show()

+----------+------------------+--------+-----+--------+----------+----+
|product_id|      product_name|group_id|price|group_id|group_name|rank|
+----------+------------------+--------+-----+--------+----------+----+
|         6|   Lenovo Thinkpad|       2|  700|       2|    Laptop|   1|
|         7|         Sony VAIO|       2|  700|       2|    Laptop|   1|
|         8|       Dell Vostro|       2|  800|       2|    Laptop|   3|
|         5|          HP Elite|       2| 1200|       2|    Laptop|   4|
|         1|   Microsoft Lumia|       1|  200|       1|Smartphone|   1|
|         2|           HTC One|       1|  400|       1|Smartphone|   2|
|         3|             Nexus|       1|  500|       1|Smartphone|   3|
|         4|            iPhone|       1|  900|       1|Smartphone|   4|
|        10|       Kindle Fire|       3|  150|       3|    Tablet|   1|
|        11|Samsung Galaxy Tab|       3|  200|       3|    Tablet|   2|
|         9|              iPad|       3|  700|       3|    Table

In [41]:
# Similar to the RANK() function, the DENSE_RANK() function assigns a rank to each row within an ordered partition, but the ranks have no gap. In other words, the same ranks are assigned to multiple rows and no ranks are skipped.

In [48]:
products.join(product_groups, products['group_id'] == product_groups['group_id']) \
.withColumn("dense_rank", F.dense_rank().over(window_with_order)).show()

+----------+------------------+--------+-----+--------+----------+----------+
|product_id|      product_name|group_id|price|group_id|group_name|dense_rank|
+----------+------------------+--------+-----+--------+----------+----------+
|         6|   Lenovo Thinkpad|       2|  700|       2|    Laptop|         1|
|         7|         Sony VAIO|       2|  700|       2|    Laptop|         1|
|         8|       Dell Vostro|       2|  800|       2|    Laptop|         2|
|         5|          HP Elite|       2| 1200|       2|    Laptop|         3|
|         1|   Microsoft Lumia|       1|  200|       1|Smartphone|         1|
|         2|           HTC One|       1|  400|       1|Smartphone|         2|
|         3|             Nexus|       1|  500|       1|Smartphone|         3|
|         4|            iPhone|       1|  900|       1|Smartphone|         4|
|        10|       Kindle Fire|       3|  150|       3|    Tablet|         1|
|        11|Samsung Galaxy Tab|       3|  200|       3|    Table

## The LAG and LEAD functions

The LAG() function has the ability to access data from the previous row, while the LEAD() function can access data from the next row.

LAG  (expression [,offset] [,default]) over_clause;

LEAD (expression [,offset] [,default]) over_clause;

 `expression` – a column or expression to compute the returned value.
 
 `offset` – the number of rows preceding ( LAG)/ following ( LEAD) the current row. It defaults to 1.
 
 `default` – the default returned value if the offset goes beyond the scope of the window. The default is NULL if you skip it.
The following statement uses the LAG() function to return the prices from the previous row and calculates the difference between the price of the current row and the previous row.

## LAG

In [59]:
products.join(product_groups, products['group_id'] == product_groups['group_id']) \
.withColumn("prev_price", F.lag("price", 1).over(window_with_order)).show()

+----------+------------------+--------+-----+--------+----------+----------+
|product_id|      product_name|group_id|price|group_id|group_name|prev_price|
+----------+------------------+--------+-----+--------+----------+----------+
|         6|   Lenovo Thinkpad|       2|  700|       2|    Laptop|      null|
|         7|         Sony VAIO|       2|  700|       2|    Laptop|       700|
|         8|       Dell Vostro|       2|  800|       2|    Laptop|       700|
|         5|          HP Elite|       2| 1200|       2|    Laptop|       800|
|         1|   Microsoft Lumia|       1|  200|       1|Smartphone|      null|
|         2|           HTC One|       1|  400|       1|Smartphone|       200|
|         3|             Nexus|       1|  500|       1|Smartphone|       400|
|         4|            iPhone|       1|  900|       1|Smartphone|       500|
|        10|       Kindle Fire|       3|  150|       3|    Tablet|      null|
|        11|Samsung Galaxy Tab|       3|  200|       3|    Table

## LEAD

In [60]:
products.join(product_groups, products['group_id'] == product_groups['group_id']) \
.withColumn("prev_price", F.lead("price", 1).over(window_with_order)).show()

+----------+------------------+--------+-----+--------+----------+----------+
|product_id|      product_name|group_id|price|group_id|group_name|prev_price|
+----------+------------------+--------+-----+--------+----------+----------+
|         6|   Lenovo Thinkpad|       2|  700|       2|    Laptop|       700|
|         7|         Sony VAIO|       2|  700|       2|    Laptop|       800|
|         8|       Dell Vostro|       2|  800|       2|    Laptop|      1200|
|         5|          HP Elite|       2| 1200|       2|    Laptop|      null|
|         1|   Microsoft Lumia|       1|  200|       1|Smartphone|       400|
|         2|           HTC One|       1|  400|       1|Smartphone|       500|
|         3|             Nexus|       1|  500|       1|Smartphone|       900|
|         4|            iPhone|       1|  900|       1|Smartphone|      null|
|        10|       Kindle Fire|       3|  150|       3|    Tablet|       200|
|        11|Samsung Galaxy Tab|       3|  200|       3|    Table